## Ground Control Station (GCS)

> A fleet simulation focused on assigning vehicles to a shared GCS.

This notebook shows how a `SimGCS` monitors a group of UAVs, launches their associated processes, and receives their telemetry. The mission and visualizer setup are supporting context; the GCS-to-vehicle relationship is the main subject.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.entities.simgcs import SimGCS
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()


## Simulation Positions

Define a common geographic origin and six relative home positions. The vehicle positions make the shared GCS scenario easier to inspect, but they do not affect GCS membership.

Each entry in `base_homes` becomes the home position for one vehicle created below.

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

base_homes = ENUPose.list(
    [  # east, north, up, heading
        (5.0, 15.0, 0.0, -45),
        (15.0, 0.0, 0.0, 0),
        (5.0, -20.0, 0.0, 30.0),
    ]
)


## Create Ground Control Station (GCS)

Create one `SimGCS` for the fleet. A GCS may monitor zero or more vehicles, and a vehicle may be monitored by more than one GCS.

The GCS membership is established when vehicles are created with `gcss=[gcs]`. That assignment updates both sides of the relationship: `vehicle.gcss` and `gcs.vehicles`. The first assigned GCS owns the vehicle's SITL, logic, ADS-B, and optional MITM processes.

In [ ]:
colors = [Color.GREEN, Color.BLUE, Color.YELLOW]
gcs = SimGCS(
    name=f"GCS_{''.join([color.emoji for color in colors])}", record_positions=True
)


## Create Vehicles

This loop creates the fleet and assigns every vehicle to the shared GCS.

1. **Fleet identity and mission**
   * `sysids`, `models`, `colors`, and `base_homes` describe the six vehicles.
   * Each vehicle receives an autonomous square mission and a relative home position.

2. **Assign the GCS**
   * `gcss=[gcs]` links the newly created vehicle to the station declared above.
   * The link is bidirectional, so `gcs.vehicles` is populated automatically.
   * To monitor a vehicle without supplying `gcss` at construction, call `veh.assign_gcs(gcs)` after creating it.

In [ ]:
sysids = range(1, 4)
models = 3 * [Model.IRIS]

side_lens = [5, 7, 3]
alt = 10
mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)
vehs: list[SimVehicle] = []

for sysid, model, base_home, color, side_len in zip(
    sysids, models, base_homes, colors, side_lens, strict=True
):
    mission_path = str(mission_folder / f"mission_{sysid}.waypoints")
    auto_plan = AutoPlan.square_traj(
        firmware=model.firmware,
        side_len=side_len,
        alt=alt,
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        mission_path=mission_path,
    )

    guided_plan = GuidedPlan.square_traj(
        side_len=side_len,
        alt=alt,
        enu_origin=enu_origin,
        relative_home=base_home,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        model=model,
        sysid=sysid,
        plan=auto_plan,
        color=color,
        gcss=[gcs],
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=Plan.create_square_path(side_len=side_len, alt=alt),
    )

    vehs.append(veh)


## Visualizer

The GCS topology is independent of visualization. Choose a visualizer for inspection: Gazebo renders the fleet in 3D, QGroundControl displays vehicle telemetry and mission state, and `NoVisualizer` runs headlessly.

For this GCS-focused example, select `qgc` in the `Simulator` cell when you want to observe the station's monitored fleet.

### Gazebo

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)
gaz.markers.append(origin_gaz)


### QGroundControl

Use the QGroundControl visualizer to inspect the fleet from the GCS perspective. Its map and telemetry views show the vehicles assigned to the shared station once the simulation is running.

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(name="origin", pos=gra_origin.unpose(), color=Color.WHITE)
qgc.markers.append(origin_qgc)


### No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)


## Simulator

The simulator registers the fleet, discovers each vehicle's assigned GCS, writes the GCS configuration, and launches its process.

1. **Register the fleet**
   * Call `orac.add_vehicle(veh)` once for every vehicle.
   * The simulator validates and registers the GCS referenced by each vehicle; you do not add this shared GCS separately in this API.

2. **Choose visible processes**
   * `terminals=[SimProcess.GCS]` opens a terminal for the GCS process while the vehicle processes run in the background.

3. **Preview before launch**
   * `simulator.preview()` renders the configured fleet without starting flight processes.

In [ ]:
orac = Oracle()

for veh in vehs:
    orac.add_vehicle(veh)

simulator = Simulator(
    oracle=orac,
    visualizer=qgc,
    terminals=[SimProcess.GCS],
    verbose=1,
)

simulator.preview()


## Run

`simulator.run()` does the whole thing: it launches the selected visualizer and
the shared GCS with its fleet's vehicle processes, binds the `Oracle` that was
built beforehand, then blocks until every vehicle and the GCS report completion.

* The Oracle coordinates Remote ID communication between the vehicles while the
  missions fly.
* The GCS terminal shows the telemetry and lifecycle activity for the assigned fleet.
* Call `simulator.launch()` and `orac.run()` separately if you want to do
  something in between.

In [ ]:
simulator.run()
